# Hermes · one-cell start

**First time only:** open the 🔑 Secrets panel and save `GIT_STATE_TOKEN` and your **existing** `STORAGE_ENCRYPTION_KEY`. Enable Notebook access for both. These are your private account secrets, not notebook contents. If missing, the launcher asks through hidden prompts.

**Every start:** stop the old Render/local/Colab copy after a successful backup, then press ▶ below. The launcher installs Hermes, restores `Sugamdeol/hermes-storage@state`, starts Telegram polling and shows two dashboard URLs (Cloudflare + Colab), the username and password together, and opens the embedded view. Providers, bot token, chats, memories and skills come from the encrypted backup. No GPU is needed. The optional Drive cache needs permission once and reuses public builds and package downloads. The first build still takes several minutes; later code updates reuse installed dependencies and dashboards.

**Root:** the agent runs as root in Colab, like notebook cells do: `!apt-get install -y libatk1.0-0 ...` and other system downloads from the agent's own tools work without sudo.

**Root gateway:** this launcher sets `HERMES_ALLOW_ROOT_GATEWAY=1`, a Colab-only opt-in. Upstream refuses to start the gateway as root inside its Docker image; Colab's VM is ephemeral and every Colab run is the root account, so that guard is accepted here. Render and Docker never set it.

Running this cell replaces typing SWITCH. Run only one host with this Telegram bot and backup repository. Colab may still ask you to connect a runtime or approve notebook execution.

Free Colab runs at most 12 hours and can end earlier. It is temporary hosting; external dashboard use can trigger early termination. [Official limits](https://research.google.com/colaboratory/faq.html).


In [ ]:
#@title ▶ Start Hermes · restore your chats, memories and settings
import hashlib
import os
use_drive_cache = True #@param {type:"boolean"}
os.environ["HERMES_COLAB_USE_DRIVE_CACHE"] = "1" if use_drive_cache else "0"
import runpy
import urllib.request
from pathlib import Path

# Running this cell starts this host; stop your other Hermes host first.
launcher_url = "https://raw.githubusercontent.com/Sugamdeol/hermes-render/c901faed35de0cd201f60120745d057aeea02728/run-colab.py"
with urllib.request.urlopen(launcher_url, timeout=60) as response:
    launcher = response.read()
if hashlib.sha256(launcher).hexdigest() != "b36e67032ea9bc0757714ed0840b07bc6d3d8af9ee77a8386e65034542d8710c":
    raise RuntimeError("Launcher download failed verification. Retry this cell; no data has been changed.")
compile(launcher, "run-colab.py", "exec")
launcher_path = Path("/content/hermes-colab-launcher.py")
launcher_path.write_bytes(launcher)
launcher_path.chmod(0o600)
namespace = runpy.run_path(str(launcher_path), run_name="hermes_colab_launcher")
launch_globals = namespace["main"].__globals__
previous = globals().get("HERMES_COLAB")
if previous is not None and type(previous).__name__ == "ColabAgent":
    previous.__class__ = namespace["ColabAgent"]
launch_globals["HERMES_COLAB"] = previous
try:
    HERMES_COLAB = namespace["main"](confirm_switch=True)
finally:
    # Keep backup/stop controls available even if startup times out.
    if launch_globals.get("HERMES_COLAB") is not None:
        HERMES_COLAB = launch_globals["HERMES_COLAB"]


### Optional controls
In a new code cell: `HERMES_COLAB.dashboard()` opens the embedded dashboard again; `HERMES_COLAB.status()` shows health and uptime; `HERMES_COLAB.password()` shows your dashboard password; `HERMES_COLAB.backup()` saves now; `HERMES_COLAB.stop()` saves and stops. If a save fails, `backup()` prints why and the agent keeps running; fix the reason, then retry. Only for a stuck runtime, where you accept losing work that has not reached GitHub, `HERMES_COLAB.stop(force=True)` stops without saving; then press ▶ to restart from the last GitHub backup.

**If the gateway refuses to start as root** ("Refusing to run the Hermes gateway as root") on a runtime started from an older copy of this notebook, and the dashboard cannot restart it: in a new cell run `with open('/opt/data/.env', 'a') as f: f.write('\nHERMES_ALLOW_ROOT_GATEWAY=1\n')`, then `HERMES_COLAB.backup()` and `HERMES_COLAB.stop()`, then press ▶ above again. Startup loads saved `.env` settings before the gateway starts. If backup fails, read the reason it prints first.
Installed pip/apt/global npm packages and Playwright browsers are saved as a recipe in your private backup and restored in the background. Check `HERMES_COLAB.dependencies()` before using extra apps. Run `HERMES_COLAB.backup()` before switching runtimes to save the latest inventory.
